# Evasion tests and adversarial training (roadmap step 9)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohaaa825/Skolfie/blob/main/notebooks/robust.ipynb)

How much do simple tricks lower detection? This notebook

* takes 1000 AI and 1000 human texts from the test split and attacks them: typos, synonym swaps,
  look-alike Unicode letters and invisible spaces, sentence-by-sentence T5 paraphrasing, and a
  splice standing in for human-edited AI text (30% of the sentences swapped for human sentences
  of the same source and topic). Noise attacks are applied to the human texts too, to check that
  they do not start being flagged;
* optionally, with an API key in the Colab secrets (`ANTHROPIC_API_KEY`, `OPENAI_API_KEY` or
  `GEMINI_API_KEY`), also LLM paraphrasing, a "humanizer" rewrite, and fresh generation at a low
  and a high temperature. Without a key these are skipped and the report says so;
* scores your saved models on every attack set and writes a robustness table per attack;
* optionally (`ADVERSARIAL_TRAINING = True`, off by default) adds attacked copies of a share of
  the training texts, retrains `transformer` as `transformer-adv`, and compares the two.

**Use a GPU runtime** (T4) for the T5 paraphraser and GPT-2 features. Then **Runtime → Run all**.
The attacks and scoring take about 30 to 45 minutes; adversarial training adds about 1.5 hours.

Writes `My Drive/skolfie/reports/robust/`.

## 1. Settings

In [ ]:
DATA_DIR = "/content/drive/MyDrive/skolfie/data/processed"
REPORTS_DIR = "/content/drive/MyDrive/skolfie/reports"
CACHE_DIR = "/content/drive/MyDrive/skolfie/cache"
MODELS_DIR = "/content/drive/MyDrive/skolfie/models"

REPO = "mohaaa825/Skolfie"
BRANCH = "main"

MODELS = ["transformer", "features"]  # saved models in MODELS_DIR to test; missing ones are skipped
N_TEXTS = 1000         # AI texts and human texts in the test sample, each
USE_LLM_ATTACKS = True # only runs if an API key is in the Colab secrets
LLM_PROVIDER = "auto"  # or "anthropic", "openai", "gemini"

ADVERSARIAL_TRAINING = False  # True: retrain the transformer with attacked copies added
BASE_MODEL = "transformer"
ADV_SHARE = 0.05       # share of the training texts that get an attacked copy
ADV_MAX_TEXTS = 8000   # at most this many attacked AI copies
ADV_ATTACKS = "t5_paraphrase,human_edit_30,typo_5,typo_15,synonym_15,homoglyph,zero_width"
ADV_PARAPHRASE_WEIGHT = 3  # paraphrase is what we most want to resist: draw it 3x as often
EPOCHS = 1
BATCH_SIZE = 16        # lower to 8 if you get a CUDA out-of-memory error

## 2. Check the GPU, mount Drive, clone and install

In [ ]:
import os
import subprocess

import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU; T5 paraphrasing and scoring will be very slow.")

from google.colab import drive

drive.mount("/content/drive")
ROBUST_DIR = f"{REPORTS_DIR}/robust"
for d in (ROBUST_DIR, CACHE_DIR, MODELS_DIR):
    os.makedirs(d, exist_ok=True)
assert os.path.exists(f"{DATA_DIR}/corpus.parquet"), "Run build_corpus.ipynb first."
assert os.path.exists(f"{DATA_DIR}/splits.parquet"), "Run evaluate.ipynb (step 3) first."


def secret(name):
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return None


def git(*args, check=True):
    """Run git; a GITHUB_TOKEN secret reaches it through a credential helper that reads it from
    the environment, so the token never appears in a URL, a command line, an error message or
    .git/config."""
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    auth = []
    token = secret("GITHUB_TOKEN")
    if token:
        env["SKOLFIE_GITHUB_TOKEN"] = token
        auth = ["-c", "credential.helper=", "-c",
                "credential.helper=!f() { test \"$1\" = get && echo username=x-access-token "
                "&& echo \"password=$SKOLFIE_GITHUB_TOKEN\"; }; f"]
    return subprocess.run(["git", *auth, *args], env=env, check=check)


url = f"https://github.com/{REPO}.git"
if os.path.isdir("/content/Skolfie/.git"):
    # Clones made by older versions of this notebook kept the token in the remote URL.
    git("-C", "/content/Skolfie", "remote", "set-url", "origin", url)
    git("-C", "/content/Skolfie", "pull", "--ff-only")
else:
    git("clone", "--depth", "1", "--branch", BRANCH, url, "/content/Skolfie")

%cd /content/Skolfie
!git log --oneline -1
!pip install -q -e ".[features]"

## 3. Build the test attack sets

Attacked texts are cached per attack in `robust/attacks-test/parts/`, so rerunning after an
interruption only redoes the missing attacks.

In [ ]:
import sys

env = dict(os.environ)
for name in ("HF_TOKEN", *(("ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GEMINI_API_KEY")
                           if USE_LLM_ATTACKS else ())):
    value = secret(name)
    if value:
        env[name] = value
print("API keys found:", [k for k in env if k.endswith("_API_KEY")] or "none (LLM attacks skipped)")


def run(cmd):
    """Run a command and stream its output into the notebook as it happens."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1, env=env)
    for line in proc.stdout:
        sys.stdout.write(line)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")


ATTACKS_TEST = f"{ROBUST_DIR}/attacks-test"
run(["python", "-u", "scripts/make_attacks.py",
     "--corpus", f"{DATA_DIR}/corpus.parquet",
     "--splits", f"{DATA_DIR}/splits.parquet",
     "--split", "test", "--n", str(N_TEXTS),
     "--provider", LLM_PROVIDER if USE_LLM_ATTACKS else "none",
     "--out-dir", ATTACKS_TEST])

## 4. Score the saved models on every attack

In [ ]:
present = [m for m in MODELS if os.path.exists(f"{MODELS_DIR}/{m}.joblib")]
print("models:", present, "| missing:", [m for m in MODELS if m not in present])
for m in present:
    run(["python", "-u", "scripts/eval_robust.py",
         "--attacks", f"{ATTACKS_TEST}/attacks.parquet",
         "--model", f"{m}={MODELS_DIR}/{m}.joblib",
         "--out-dir", f"{ROBUST_DIR}/{m}"])

## 5. Adversarial training (optional)

Runs only with `ADVERSARIAL_TRAINING = True`. Builds attacked copies of a share of the training
texts (noise attacks on human texts too, in the same proportion, so noise is no label signal),
retrains the transformer with them as `transformer-adv`, scores it on the held-out splits, and
compares both models on the same test attack sets. Done when the AUROC loss under paraphrase is
at least halved while the FPR on clean human text stays the same.

In [ ]:
ADV_MODEL = f"{BASE_MODEL}-adv"
if not ADVERSARIAL_TRAINING:
    print("ADVERSARIAL_TRAINING is False: skipped.")
else:
    ATTACKS_TRAIN = f"{ROBUST_DIR}/attacks-train"
    run(["python", "-u", "scripts/make_attacks.py",
         "--corpus", f"{DATA_DIR}/corpus.parquet",
         "--splits", f"{DATA_DIR}/splits.parquet",
         "--split", "train", "--share", str(ADV_SHARE), "--n", str(ADV_MAX_TEXTS),
         "--attacks", ADV_ATTACKS,
         "--weight", f"t5_paraphrase={ADV_PARAPHRASE_WEIGHT}",
         "--out-dir", ATTACKS_TRAIN])
    run(["python", "-u", "scripts/evaluate.py",
         "--corpus", f"{DATA_DIR}/corpus.parquet",
         "--splits", f"{DATA_DIR}/splits.parquet",
         "--model", BASE_MODEL,
         "--extra-train", f"{ATTACKS_TRAIN}/attacks.parquet",
         "--out-dir", f"{REPORTS_DIR}/{ADV_MODEL}",
         "--cache-dir", CACHE_DIR,
         "--epochs", str(EPOCHS),
         "--batch-size", str(BATCH_SIZE),
         "--save-model", f"{MODELS_DIR}/{ADV_MODEL}.joblib",
         "--save-predictions"])
    run(["python", "-u", "scripts/eval_robust.py",
         "--attacks", f"{ATTACKS_TEST}/attacks.parquet",
         "--model", f"{BASE_MODEL}={MODELS_DIR}/{BASE_MODEL}.joblib",
         "--model", f"{ADV_MODEL}={MODELS_DIR}/{ADV_MODEL}.joblib",
         "--out-dir", f"{ROBUST_DIR}/{BASE_MODEL}-vs-adv"])
    run(["python", "scripts/evaluate.py", "--compare",
         f"{REPORTS_DIR}/{BASE_MODEL}/report.json", f"{REPORTS_DIR}/{ADV_MODEL}/report.json"])

Reports are in `My Drive/skolfie/reports/robust/` (one folder per model, plus
`transformer-vs-adv/` after adversarial training). Tell Claude in the project thread when this
has finished; it reads the report from Drive directly.